# NB3 - Canonical latency, KAN edge statistics, annotation kit, statistics, final table, readiness gate (Kaggle T4)

**Settings:** Accelerator = GPU T4 (needed only for the latency cell), Internet = ON.  **Run:** Run All.

**Datasets to attach (Add Data):**
1. `KAN_ROUTE_PART4_JOURNAL_READY.zip` (same dataset as NB1/NB2)
2. The **output of NB1** (Add Data > Notebook Output > your NB1 notebook) - or the NB1 folders uploaded as a dataset: it must contain `00_frozen_baseline/`, `01_canonical_feature_ablation/`, `02_parameter_matched_mlp/`, `03_linear_baseline/` and `all_cells_by_seed_NB1.csv`
3. The **output of NB2** (it must contain `04_zero_shot_llm/`)
4. `images-new-known` (folders `known/` and `new/`) - for real-image latency and the annotation image folder

Not needed: `kan-qwen`, `llama-kan`.

**What it does:** re-verifies the canonical evidence is byte-identical to the NB1 freeze; measures canonical latency on real human queries + real images (4 heads x 4 modes, T4, batch 1, inference mode); verifies the KAN edge-function statistics from the 9 canonical checkpoints; builds the independent-annotation kit (blinded CSV, flat image folder, guide, templates) and computes agreement automatically if `human_annotation_annotator*.csv` files are attached later; runs paired statistics (per-seed McNemar, paired bootstrap, Holm) incl. zero-shot LLMs vs trained heads; builds the final comparison table, the claim-changes list and the readiness gate; and merges NB1+NB2+NB3 into `/kaggle/working/KAN_ROUTE_NEUROCOMPUTING_FINAL_EXPERIMENTS.zip`.

The gate will report `NEUROCOMPUTING_SUBMISSION_EVIDENCE_READY = NO` until independent annotation + adjudication exist - by design.

In [ ]:
# ===== NB3 | Setup (GPU only needed for the latency cell) =====
import os, sys, re, json, time, math, shutil, random, subprocess, warnings, gc, hashlib, ast
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
def pip_install(*pk): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pk], check=False)
pip_install("open_clip_torch", "sentence-transformers", "tabulate")
import torch, torch.nn as nn
assert torch.cuda.is_available(), "Turn on a GPU (T4): the latency cell needs it."
GPU_NAME = torch.cuda.get_device_name(0); print("GPU:", GPU_NAME, "| torch", torch.__version__)
FASTKAN_DIR = Path("/kaggle/working/fast-kan")
if not FASTKAN_DIR.exists(): subprocess.run(["git", "clone", "https://github.com/ZiyaoLi/fast-kan.git", str(FASTKAN_DIR)], check=True)
sys.path.append(str(FASTKAN_DIR))
from fastkan import FastKAN

def find_jr_base():
    for p in Path("/kaggle/input").rglob("PART4_FINAL_REPRODUCIBILITY_AUDIT.md"):
        d = p.parent
        if (d / "part4_predictions_seed_corrected").is_dir() and (d / "qwen_shared28_train.csv").exists(): return d
    raise FileNotFoundError("Attach KAN_ROUTE_PART4_JOURNAL_READY.zip as a dataset.")
def find_dir_named(name):
    hits = [p for p in Path("/kaggle/input").rglob(name) if p.is_dir()]
    if not hits: raise FileNotFoundError(f"Attach the output of the notebook that produced '{name}' (Add Data > Notebook Output).")
    return hits[0]
JR = find_jr_base()
D00_NB1, D01, D02, D03 = [find_dir_named(n) for n in ("00_frozen_baseline", "01_canonical_feature_ablation", "02_parameter_matched_mlp", "03_linear_baseline")]
D04 = find_dir_named("04_zero_shot_llm")
ALLCELLS = next(Path("/kaggle/input").rglob("all_cells_by_seed_NB1.csv"))   # NB1 root-level results file
NB1_ROOT = ALLCELLS.parent
print("JR:", JR, "\nNB1 results:", ALLCELLS, "\nNB2:", D04)
OUT = Path("/kaggle/working/KAN_ROUTE_NEUROCOMPUTING_FINAL_EXPERIMENTS")
D05, D06, D07, D08, D09 = [OUT / n for n in ("05_canonical_latency", "06_human_validation", "07_interpretability", "08_statistics", "09_final")]
for d in (D05, D06, D07, D08, D09): d.mkdir(parents=True, exist_ok=True)
SEEDS = [42, 123, 2026]; MLP_PM_PROJ = 61


In [ ]:
# ======================= LIB_EVAL (pure numpy/pandas/sklearn; no torch) =======================
import json, re, math, hashlib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score

SEEDS = [42, 123, 2026]
N_CLASSES = 28
ALL_LABELS = list(range(N_CLASSES))

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def normalize_q(q):
    return re.sub(r"[^\w\s]", "", str(q).lower()).strip()

def macro_f1_fast(y, p, labels):
    """Macro-F1 over `labels` (zero-division -> 0). Equivalent to sklearn f1_score(average='macro', labels=labels)."""
    labels = np.asarray(labels)
    k = int(max(y.max(), p.max(), labels.max())) + 1
    tp = np.bincount(y[y == p], minlength=k).astype(float)
    pc = np.bincount(p, minlength=k).astype(float)
    yc = np.bincount(y, minlength=k).astype(float)
    denom = pc + yc
    f1 = np.divide(2 * tp, denom, out=np.zeros(k), where=denom > 0)
    return float(f1[labels].mean())

# ---------------------------------- prediction files ----------------------------------
def read_pred(path):
    df = pd.read_csv(path)
    df["_top3"] = [json.loads(s) for s in df["pred_top3"]]
    return df

def top3_hit(y, top3_list):
    return np.fromiter((g in t for g, t in zip(y, top3_list)), bool, len(y))

def metrics_from_arrays(y, p, hit3, labels):
    if len(y) == 0:
        return dict(n=0, top1=np.nan, top3=np.nan, macro_f1=np.nan, weighted_f1=np.nan)
    return dict(n=int(len(y)), top1=float((y == p).mean()), top3=float(hit3.mean()),
                macro_f1=macro_f1_fast(y, p, labels),
                weighted_f1=float(f1_score(y, p, average="weighted", labels=labels, zero_division=0)))

# ---------------------------------- human benchmark ----------------------------------
class HumanBench:
    """Loads human_shared28.csv and derives all strata. Fixes the NaN->'nan' bug (acceptable set empty for strict rows)."""
    def __init__(self, human_csv, qtr_csv, ltr_csv, mapping_json):
        h = pd.read_csv(human_csv).reset_index(drop=True)
        self.map = json.load(open(mapping_json))["class_to_local_id"]
        self.inv = {v: k for k, v in self.map.items()}; self.inv[N_CLASSES] = "__INVALID__"
        h["acceptable_tool_labels"] = h["acceptable_tool_labels"].fillna("").astype(str).str.strip()
        strict_true = h["strict_eval"].astype(str).str.lower().isin(["true", "1"]).values
        self.is_amb = strict_true & (h["acceptable_tool_labels"] != "").values
        self.is_strict = strict_true & ~self.is_amb
        assert self.is_strict.sum() + self.is_amb.sum() == len(h)
        self.h = h
        qtr, ltr = pd.read_csv(qtr_csv), pd.read_csv(ltr_csv)
        train_imgs = set(qtr.img_id) | set(ltr.img_id)
        self.known = h["image_id"].isin(train_imgs).values
        assert (self.known == (h["image_pool"] == "known").values).all(), "known/new recomputation disagrees with image_pool"
        self.vis_yes = (h["routing_requires_visual_context"].astype(str).str.lower() == "yes").values
        self.y = h["tool_name"].map(self.map).values.astype(int)
        self.acc_sets = [set(x.strip() for x in s.split(",") if x.strip()) for s in h["acceptable_tool_labels"]]
        self.strata = {"all": np.ones(len(h), bool), "vis_yes": self.vis_yes, "vis_no": ~self.vis_yes,
                       "known": self.known, "new": ~self.known}
        shared = sorted(set(self.y[self.known]) & set(self.y[~self.known]))
        self.shared_labels = shared

    def eval_human(self, df, use_bench_labels=False):
        assert len(df) == len(self.h) and (df["sample_id"].astype(str).values == self.h["query_id"].astype(str).values).all(), \
            "human prediction rows not aligned with query_id"
        y, p, t3 = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int), df["_top3"].values
        if use_bench_labels:
            y = self.y.copy()
        else:
            assert (y == self.y).all()
        hit3 = top3_hit(y, t3)
        top1_names = [self.inv[i] for i in p]
        amb1 = np.array([top1_names[i] in self.acc_sets[i] for i in range(len(y))])
        amb3 = np.array([bool({self.inv[c] for c in t3[i]} & self.acc_sets[i]) for i in range(len(y))])
        rows = []
        for name, m in self.strata.items():
            labs = ALL_LABELS if name == "all" else sorted(set(y[m]))
            r = metrics_from_arrays(y[m], p[m], hit3[m], labs)
            r["macro_f1_label_set"] = "28" if name == "all" else "present"
            if name in ("known", "new"):
                r["macro_f1_shared26"] = macro_f1_fast(y[m], p[m], self.shared_labels)
            sm, am = m & self.is_strict, m & self.is_amb
            r.update(strict_n=int(sm.sum()), strict_top1=float((y[sm] == p[sm]).mean()) if sm.any() else np.nan,
                     strict_top3=float(hit3[sm].mean()) if sm.any() else np.nan,
                     ambig_n=int(am.sum()), ambig_top1=float(amb1[am].mean()) if am.any() else np.nan,
                     ambig_top3=float(amb3[am].mean()) if am.any() else np.nan)
            r["cell"] = f"human_{name}"
            rows.append(r)
        return rows

class SyntheticBench:
    """Image-disjoint masks for the two teacher test sets."""
    def __init__(self, qtr_csv, qte_csv, ltr_csv, lte_csv):
        self.qtr, self.qte, self.ltr, self.lte = [pd.read_csv(p) for p in (qtr_csv, qte_csv, ltr_csv, lte_csv)]
        qkeys, lkeys = set(zip(self.qtr.img_id, self.qtr.tool)), set(zip(self.ltr.img_id, self.ltr.tool))
        self.disj = {"qwen_test": np.array([(i, t) not in lkeys for i, t in zip(self.qte.img_id, self.qte.tool)]),
                     "llama_test": np.array([(i, t) not in qkeys for i, t in zip(self.lte.img_id, self.lte.tool)])}

    def eval_synth(self, df, which):
        y, p, t3 = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int), df["_top3"].values
        hit3 = top3_hit(y, t3)
        out = []
        for suffix, m in (("", np.ones(len(y), bool)), ("_disjoint", self.disj[which])):
            r = metrics_from_arrays(y[m], p[m], hit3[m], ALL_LABELS)
            r["cell"] = which + suffix
            out.append(r)
        return out

def eval_prediction_set(preds, sbench, hbench):
    """preds: dict with keys qwen_test, llama_test, human -> prediction DataFrames. Returns list of metric rows (cells)."""
    rows = []
    rows += sbench.eval_synth(preds["qwen_test"], "qwen_test")
    rows += sbench.eval_synth(preds["llama_test"], "llama_test")
    rows += hbench.eval_human(preds["human"])
    return rows

def per_class_f1(df):
    y, p = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int)
    k = N_CLASSES
    tp = np.bincount(y[y == p], minlength=k).astype(float); pc = np.bincount(p, minlength=k).astype(float); yc = np.bincount(y, minlength=k).astype(float)
    d = pc + yc
    return np.divide(2 * tp, d, out=np.zeros(k), where=d > 0), yc.astype(int)

# ---------------------------------- aggregation ----------------------------------
def sd(x):
    x = np.asarray(x, float)
    return float(np.std(x, ddof=1)) if len(x) > 1 else 0.0

def aggregate_seeds(rows_df, keys, metrics):
    out = []
    for k, g in rows_df.groupby(keys, sort=False):
        rec = dict(zip(keys, k if isinstance(k, tuple) else (k,)))
        for m in metrics:
            v = g.sort_values("seed")[m].values.astype(float)
            rec[f"{m}_mean"], rec[f"{m}_sd"] = float(np.nanmean(v)), sd(v)
            for s, val in zip(sorted(g["seed"].values), v):
                rec[f"{m}_seed{s}"] = float(val)
        rec["n_seeds"] = int(g["seed"].nunique())
        out.append(rec)
    return pd.DataFrame(out)

def paired_delta(rows_df, id_cols, factor, a, b, metrics):
    """Seed-paired delta = metric(factor==a) - metric(factor==b); returns per-seed rows."""
    A = rows_df[rows_df[factor] == a].set_index(id_cols + ["seed"])
    B = rows_df[rows_df[factor] == b].set_index(id_cols + ["seed"])
    idx = A.index.intersection(B.index)
    d = pd.DataFrame(index=idx)
    for m in metrics:
        d[m] = A.loc[idx, m].values - B.loc[idx, m].values
    return d.reset_index()

# ---------------------------------- statistics ----------------------------------
def mcnemar_exact(correct_a, correct_b):
    from scipy.stats import binomtest
    b = int((correct_a & ~correct_b).sum()); c = int((~correct_a & correct_b).sum())
    n = b + c
    p = 1.0 if n == 0 else float(binomtest(min(b, c), n, 0.5, alternative="two-sided").pvalue)
    return b, c, p

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def paired_bootstrap(preds_a, preds_b, y, B=1000, seed=0, labels=ALL_LABELS):
    """preds_*: list over seeds of int arrays on identical rows. Returns CIs of seed-averaged delta Top-1 and delta macro-F1."""
    rng = np.random.RandomState(seed); n = len(y)
    d1, df1 = [], []
    for _ in range(B):
        idx = rng.randint(0, n, n); yy = y[idx]
        t = np.mean([(pa[idx] == yy).mean() - (pb[idx] == yy).mean() for pa, pb in zip(preds_a, preds_b)])
        f = np.mean([macro_f1_fast(yy, pa[idx], labels) - macro_f1_fast(yy, pb[idx], labels) for pa, pb in zip(preds_a, preds_b)])
        d1.append(t); df1.append(f)
    q = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return q(d1), q(df1)

# ---------------------------------- KAN edge statistics ----------------------------------
def kan_edge_stats(spline_w, in_dim, G=4, lo=-2.0, hi=2.0, npts=200):
    """spline_w: (out_dim, in_dim*G) FastKAN spline_linear weight. Returns per-edge arrays."""
    out_dim = spline_w.shape[0]
    centers = np.linspace(lo, hi, G); rho = (hi - lo) / (G - 1)
    v = np.linspace(lo, hi, npts)
    Bm = np.exp(-(((v[:, None] - centers[None, :]) / rho) ** 2))               # (npts,G)
    W = spline_w.reshape(out_dim, in_dim, G)
    S = np.einsum("pg,oig->oip", Bm, W).reshape(-1, npts)                       # (edges,npts)
    amp = S.max(1) - S.min(1)
    vc = v - v.mean(); sxx = (vc ** 2).sum()
    slope = (S - S.mean(1, keepdims=True)) @ vc / sxx
    fit_ss = (slope ** 2) * sxx
    tot_ss = ((S - S.mean(1, keepdims=True)) ** 2).sum(1)
    r2 = np.divide(fit_ss, tot_ss, out=np.ones_like(fit_ss), where=tot_ss > 1e-12)
    dS = np.diff(S, axis=1)
    mono = (dS >= -1e-9).all(1) | (dS <= 1e-9).all(1)
    return dict(amp=amp.reshape(out_dim, in_dim), r2=r2, mono=mono, out_dim=out_dim, in_dim=in_dim)

def edge_summary(st, topks=(3, 10)):
    r = dict(edges=int(st["r2"].size), near_linear_share=float((st["r2"] >= 0.95).mean()), median_r2=float(np.median(st["r2"])),
             monotone_share=float(st["mono"].mean()), median_amp=float(np.median(st["amp"])))
    A = st["amp"]
    for k in topks:
        if A.shape[1] > k:
            srt = -np.sort(-A, axis=1)
            r[f"top{k}_amp_share"] = float((srt[:, :k].sum(1) / srt.sum(1)).mean())
    return r

# ---------------------------------- LaTeX helpers ----------------------------------
def pm(mu, s, scale=100.0, nd=2):
    return f"{mu*scale:.{nd}f}$\\pm${s*scale:.{nd}f}"

def latex_table(df, caption, label, colfmt=None):
    cols = list(df.columns); colfmt = colfmt or ("l" * len(cols))
    esc = lambda x: str(x).replace("_", "\\_").replace("%", "\\%")
    L = ["\\begin{table}[t]", "\\centering", "\\small", f"\\begin{{tabular}}{{{colfmt}}}", "\\toprule",
         " & ".join(esc(c) for c in cols) + " \\\\", "\\midrule"]
    for _, r in df.iterrows():
        L.append(" & ".join(v if "$\\pm$" in str(v) else esc(v) for v in r.values) + " \\\\")
    L += ["\\bottomrule", "\\end{tabular}", f"\\caption{{{caption}}}", f"\\label{{{label}}}", "\\end{table}"]
    return "\n".join(L)

# ---------------------------------- annotation agreement ----------------------------------
def jaccard(a, b):
    a, b = set(a), set(b)
    return 1.0 if not a and not b else len(a & b) / len(a | b)

def parse_set(s):
    return sorted({x.strip() for x in str(s).replace(";", ",").split(",") if x.strip() and x.strip().lower() != "nan"})

def agreement_report(a, b, tool_col="chosen_tool", amb_col="is_ambiguous", vis_col="routing_requires_visual_context", set_col="acceptable_tool_labels"):
    from sklearn.metrics import cohen_kappa_score
    m = a.merge(b, on="query_id", suffixes=("_1", "_2"))
    res = {"n_items": len(m)}
    res["tool_raw_agreement"] = float((m[f"{tool_col}_1"] == m[f"{tool_col}_2"]).mean())
    res["tool_cohen_kappa"] = float(cohen_kappa_score(m[f"{tool_col}_1"], m[f"{tool_col}_2"]))
    for nm, col in (("ambiguity", amb_col), ("visual_context", vis_col)):
        x, y = m[f"{col}_1"].astype(str).str.lower().str.strip(), m[f"{col}_2"].astype(str).str.lower().str.strip()
        res[f"{nm}_raw_agreement"] = float((x == y).mean())
        res[f"{nm}_cohen_kappa"] = float(cohen_kappa_score(x, y)) if x.nunique() > 1 or y.nunique() > 1 else float("nan")
    s1 = [parse_set(v) for v in m[f"{set_col}_1"]]; s2 = [parse_set(v) for v in m[f"{set_col}_2"]]
    jac = np.array([jaccard(u, v) for u, v in zip(s1, s2)])
    res["acceptable_set_exact_agreement"] = float(np.mean([u == v for u, v in zip(s1, s2)]))
    res["acceptable_set_mean_jaccard"] = float(jac.mean()); res["acceptable_set_median_jaccard"] = float(np.median(jac))
    return res, m


# ---------------------------------- stable-support (21 classes) ----------------------------------
def stable_metrics(y, p, hit3, stable_ids):
    """Only queries whose gold class is stable; F1 averaged over the stable classes; predictions into other classes are errors."""
    m = np.isin(y, stable_ids)
    if not m.any():
        return dict(n=0, top1=np.nan, top3=np.nan, macro_f1=np.nan)
    return dict(n=int(m.sum()), top1=float((y[m] == p[m]).mean()), top3=float(hit3[m].mean()),
                macro_f1=macro_f1_fast(y[m], p[m], np.asarray(stable_ids)))

def add_stable_cells(rows, preds, stable_ids):
    for key, cell in (("qwen_test", "qwen_test_stable21"), ("llama_test", "llama_test_stable21"), ("human", "human_all_stable21")):
        df = preds[key]; y, p = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int)
        r = stable_metrics(y, p, top3_hit(y, df["_top3"].values), stable_ids); r["cell"] = cell; rows.append(r)
    return rows

# ---------------------------------- pooled evaluation groups (as in manuscript Table 13) ----------------------------------
GROUPS = {
    "within_teacher": [("qwen", "qwen_test"), ("llama", "llama_test"), ("mixed", "qwen_test"), ("mixed", "llama_test")],
    "cross_teacher": [("qwen", "llama_test"), ("llama", "qwen_test")],
    "image_disjoint": [("qwen", "llama_test_disjoint"), ("mixed", "llama_test_disjoint"), ("llama", "qwen_test_disjoint"), ("mixed", "qwen_test_disjoint")],
    "human_written": [("qwen", "human_all"), ("llama", "human_all"), ("mixed", "human_all")],
    "human_new_image": [("qwen", "human_new"), ("llama", "human_new"), ("mixed", "human_new")],
}

def pooled_group(rows_df, model_col, model, group, metric, groups=None):
    groups = groups or GROUPS
    """Per-seed mean over the group's (regime, cell) pairs -> array over SEEDS."""
    out = []
    for s in SEEDS:
        v = []
        for reg, cell in groups[group]:
            r = rows_df[(rows_df[model_col] == model) & (rows_df["regime"] == reg) & (rows_df["cell"] == cell) & (rows_df["seed"] == s)]
            v.append(float(r[metric].iloc[0]))
        out.append(np.mean(v))
    return np.array(out)

def pooled_compare(rows_df, model_col, a, b, metrics=("top1", "top3", "macro_f1"), groups=None):
    groups = groups or GROUPS
    res = []
    for g in groups:
        for m in metrics:
            va, vb = pooled_group(rows_df, model_col, a, g, m, groups), pooled_group(rows_df, model_col, b, g, m, groups)
            d = va - vb
            res.append(dict(group=g, metric=m, a=a, b=b, a_mean=va.mean(), a_sd=sd(va), b_mean=vb.mean(), b_sd=sd(vb),
                            diff_mean=d.mean(), diff_sd=sd(d), seeds_a_ahead=int((d > 0).sum())))
    return pd.DataFrame(res)

# ---------------------------------- zero-shot LLM output parsing ----------------------------------
REFUSAL_PAT = re.compile(r"(i cannot|i can't|i can not|i'm sorry|i am sorry|unable to|none of the (above|tools)|no (suitable|appropriate) tool|cannot determine)", re.I)

def _norm_tok(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def parse_tool(text, tools):
    """Deterministic parser. Returns (tool_or_None, status) with status in valid_exact/valid_parsed/multiple/refusal/invalid."""
    t = _norm_tok(text)
    if t in tools:
        return t, "valid_exact"
    names = sorted(tools, key=len, reverse=True)
    rest, found = f"_{t}_", []
    for n in names:                                     # longest first so a short name inside a longer one is not double-counted
        if f"_{n}_" in rest:
            found.append(n); rest = rest.replace(f"_{n}_", "__")
    if len(found) == 1:
        return found[0], "valid_parsed"
    if len(found) > 1:
        return None, "multiple"
    return None, ("refusal" if REFUSAL_PAT.search(str(text)) else "invalid")

def parse_top3(text, tools):
    """Ordered list of up to 3 distinct valid tool names in order of appearance."""
    found = []
    for part in re.split(r"[,\n;]| and ", str(text)):
        tool, st = parse_tool(part, tools)
        if tool is not None and tool not in found:
            found.append(tool)
    if not found:                                       # fall back to order-of-appearance scan of the whole text
        t = f"_{_norm_tok(text)}_"
        pos = sorted((t.find(f"_{n}_"), n) for n in tools if f"_{n}_" in t)
        found = [n for _, n in pos]
    return found[:3]


GROUPS_STABLE = {
    "within_teacher": [(r, c + "_stable21") for r, c in GROUPS["within_teacher"]],
    "cross_teacher": [(r, c + "_stable21") for r, c in GROUPS["cross_teacher"]],
    "human_written": [(r, "human_all_stable21") for r in ("qwen", "llama", "mixed")],
}

def latex_table_resized(df, caption, label):
    t = latex_table(df, caption, label)
    return t.replace("\\begin{tabular}", "\\resizebox{\\textwidth}{!}{%\n\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}}", 1)


# Written once, BEFORE any zero-shot LLM or independent-annotation results were seen. Shared by NB2 (LLM prompt) and NB3 (annotation guide).
TOOL_DEFS = {
 "3d_reconstruction": "Build a 3D model, mesh or point cloud of the scene or of an object from the image.",
 "action_recognition": "Identify what action or activity people or animals are performing (running, cooking, riding).",
 "age_gender_estimation": "Estimate the age and/or gender of the people in the image.",
 "animal_detection": "Find and locate animals in the image (which animals and where).",
 "anomaly_detection": "Find unusual, abnormal, defective or out-of-place objects or regions in the image.",
 "attribute_recognition": "Recognize attributes of an object or person such as colour, material, clothing, texture or state.",
 "background_removal": "Remove or replace the background and keep the foreground subject as a cut-out.",
 "depth_estimation": "Estimate depth or distance from the camera (how far things are, which is closer).",
 "face_detection": "Detect and locate human faces in the image.",
 "facial_expression_recognition": "Recognize the emotion or expression on a person's face (smiling, angry, surprised).",
 "fine_grained_classification": "Identify the specific subtype of an object (dog breed, bird species, car model, plant variety).",
 "image_captioning": "Write a general natural-language description (caption) of the whole image.",
 "image_classification": "Assign the whole image a single overall category or label (what kind of image / main object class).",
 "instance_segmentation": "Segment each individual object separately with its own pixel mask.",
 "logo_detection": "Detect and recognize brand logos in the image.",
 "object_counting": "Count how many objects of a given kind are in the image.",
 "object_detection": "Detect and locate objects of various categories with bounding boxes.",
 "person_detection": "Detect and locate people (pedestrians) in the image.",
 "pose_estimation": "Estimate the body pose of people (skeleton / joint keypoints, posture).",
 "relation_detection": "Identify relationships between objects or people (person riding horse, cup on table).",
 "salient_object_segmentation": "Segment the most visually prominent object(s) in the image.",
 "scene_recognition": "Recognize the type of scene or place (kitchen, beach, street, indoor or outdoor).",
 "semantic_segmentation": "Label every pixel with a semantic class (road, sky, building) without separating instances.",
 "style_transfer": "Re-render the image in the style of an artwork, an artist or another visual style.",
 "super_resolution": "Increase resolution, sharpen, denoise or enhance a low-quality image.",
 "vehicle_detection": "Detect and locate vehicles (cars, trucks, buses, motorbikes, bicycles).",
 "visual_question_answering": "Answer a specific factual question about the content of the image.",
 "visual_reasoning": "Answer questions that need reasoning over the image: spatial or logical relations, comparison, inference about what is happening or why.",
}
BOUNDARY_NOTES = [
 "image_captioning vs visual_question_answering: captioning describes the whole image; VQA answers one specific question.",
 "image_classification vs object_detection vs fine_grained_classification: classification labels the whole image; detection locates objects; fine-grained names the specific subtype.",
 "semantic_segmentation vs instance_segmentation vs salient_object_segmentation: semantic labels every pixel by class; instance separates each object; salient picks the most prominent object.",
 "person_detection vs face_detection vs pose_estimation: whole people; faces only; body joints/posture.",
 "visual_question_answering vs visual_reasoning: a direct factual lookup vs a question needing spatial/logical inference.",
 "scene_recognition vs image_classification vs image_captioning: type of place; overall category; free-text description.",
 "depth_estimation vs 3d_reconstruction: distance/ordering of things in view vs building a 3D model.",
]
assert len(TOOL_DEFS) == 28


In [ ]:
# ===== NB3 | Head definitions (identical to NB1/canonical) + checkpoint reader =====
class KANRoute(nn.Module):
    def __init__(self, input_dim=896, proj_dim=48, hidden_dims=(48,), num_tools=28, num_grids=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim); self.norm = nn.LayerNorm(proj_dim)
        self.kan = FastKAN(layers_hidden=[proj_dim] + list(hidden_dims) + [num_tools], num_grids=num_grids)
    def forward(self, x): return self.kan(self.norm(self.proj(x.float())))
class MLPBaseline(nn.Module):
    def __init__(self, input_dim=896, proj_dim=96, hidden_dims=(64, 32), num_tools=28, dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, proj_dim), nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(dropout)]; prev = proj_dim
        for h in hidden_dims: layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]; prev = h
        layers.append(nn.Linear(prev, num_tools)); self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x.float())
class LinearHead(nn.Module):
    def __init__(self, input_dim=896, num_tools=28): super().__init__(); self.fc = nn.Linear(input_dim, num_tools)
    def forward(self, x): return self.fc(x.float())
def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
def build_model(arch, input_dim=896):
    return {"kan": lambda: KANRoute(input_dim=input_dim), "mlp": lambda: MLPBaseline(input_dim=input_dim),
            "mlp_pm": lambda: MLPBaseline(input_dim=input_dim, proj_dim=MLP_PM_PROJ), "linear": lambda: LinearHead(input_dim=input_dim)}[arch]()
def load_sd_np(p):
    sd = torch.load(p, map_location="cpu"); return {k: v.numpy() for k, v in sd.items()}


In [ ]:
# ===== NB3 | Re-verify that the canonical evidence is byte-identical to the NB1 freeze, and that leakage is still zero =====
man = (D00_NB1 / "00_sha256_manifest.txt").read_text().splitlines()
pairs = [l.split("  ", 1) for l in man if l.strip()]
mism = [rel for sha, rel in pairs if (not (JR / rel).exists()) or sha256_file(JR / rel) != sha]
mp = json.load(open(JR / "shared_28_class_mapping.json"))["class_to_local_id"]
qtr, qva, qte = [pd.read_csv(JR / f"qwen_shared28_{s}.csv") for s in ("train", "val", "test")]
ltr, lva, lte = [pd.read_csv(JR / f"llama_shared28_{s}.csv") for s in ("train", "val", "test")]
def overlaps(a, b, c):
    n = [set(d["query"].map(normalize_q)) for d in (a, b, c)]; it = [set(zip(d.img_id, d.tool)) for d in (a, b, c)]
    return (len(n[0] & n[1]) + len(n[0] & n[2]) + len(n[1] & n[2]), len(it[0] & it[1]) + len(it[0] & it[2]) + len(it[1] & it[2]))
qo, lo = overlaps(qtr, qva, qte), overlaps(ltr, lva, lte)
RECHECK = dict(files_checked=len(pairs), hash_mismatches=mism, canonical_unchanged=not mism, n_classes=len(mp), matting_absent="matting" not in mp,
               qwen_overlaps=list(qo), llama_overlaps=list(lo), leakage_zero=(qo == (0, 0) and lo == (0, 0)))
json.dump(RECHECK, open(D09 / "frozen_baseline_recheck.json", "w"), indent=2); print(json.dumps(RECHECK, indent=2))
assert RECHECK["canonical_unchanged"] and RECHECK["leakage_zero"], "Canonical evidence changed or leakage detected - stop."
hb = HumanBench(JR / "human_shared28.csv", JR / "qwen_shared28_train.csv", JR / "llama_shared28_train.csv", JR / "shared_28_class_mapping.json")
sb = SyntheticBench(JR / "qwen_shared28_train.csv", JR / "qwen_shared28_test.csv", JR / "llama_shared28_train.csv", JR / "llama_shared28_test.csv")
TOOL2ID = hb.map; ID2TOOL = {v: k for k, v in TOOL2ID.items()}; TOOLS = set(TOOL2ID)


In [ ]:
# ===== NB3 | Section 7: canonical latency on REAL human queries and REAL images, T4, batch 1, inference mode =====
from sentence_transformers import SentenceTransformer
import open_clip
from PIL import Image
def find_img_dirs():
    k = [p for p in Path("/kaggle/input").rglob("known") if p.is_dir() and any(p.glob("*.jpg"))]
    n = [p for p in Path("/kaggle/input").rglob("new") if p.is_dir() and any(p.glob("*.jpg"))]
    if not k or not n: raise FileNotFoundError("Attach the images dataset with known/ and new/ folders.")
    return k[0], n[0]
KNOWN_DIR, NEW_DIR = find_img_dirs()
def img_path(i):
    for d in (KNOWN_DIR, NEW_DIR):
        p = d / f"{i}.jpg"
        if p.exists(): return p
    raise FileNotFoundError(i)
dev = torch.device("cuda")
minilm = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
clip_model, _, clip_pre = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai", force_quick_gelu=True); clip_model = clip_model.to(dev).eval()

N_PAIRS, N_WARM, N_TIMED = 200, 50, 500
sub = hb.h.sample(n=N_PAIRS, random_state=0).reset_index(drop=True)
QUERIES, PATHS = sub["query"].tolist(), [img_path(i) for i in sub.image_id]
@torch.inference_mode()
def enc_text(q): return minilm.encode([q], convert_to_tensor=True, normalize_embeddings=False, show_progress_bar=False).to(dev).float()
@torch.inference_mode()
def enc_img_tensor(x): f = clip_model.encode_image(x); return (f / f.norm(dim=-1, keepdim=True)).float()
PRE = [clip_pre(Image.open(p).convert("RGB")).unsqueeze(0).to(dev) for p in PATHS]             # decoded+preprocessed once (for the compute-only mode)
CLIPV = [enc_img_tensor(x) for x in PRE]; TXTV = [enc_text(q) for q in QUERIES]
X_PRE = [torch.cat([t, c], 1) for t, c in zip(TXTV, CLIPV)]

CK = JR / "checkpoints_seed_corrected"
HEAD_CK = {"kan": CK / "kan_mixed_shared28_seed42_corrected_best.pt", "mlp": CK / "mlp_mixed_shared28_seed42_corrected_best.pt",
           "mlp_pm": D02 / "checkpoints" / "text_pixel_mlp_pm_mixed_seed42_best.pt", "linear": D03 / "checkpoints" / "text_pixel_linear_mixed_seed42_best.pt"}
HEADS, PARAMS = {}, {}
for a, p in HEAD_CK.items():
    m = build_model(a).to(dev); m.load_state_dict(torch.load(p, map_location=dev)); m.eval(); HEADS[a] = m; PARAMS[a] = n_params(m)
print(PARAMS)

def timeit(fn):
    with torch.inference_mode():
        for i in range(N_WARM): fn(i % N_PAIRS)
        ts = []
        for i in range(N_TIMED):
            torch.cuda.synchronize(); t0 = time.perf_counter(); fn(i % N_PAIRS); torch.cuda.synchronize(); ts.append((time.perf_counter() - t0) * 1000)
    return np.array(ts)
def modes(head):
    return {
     "router_only": lambda i: head(X_PRE[i]),
     "cached_image": lambda i: head(torch.cat([enc_text(QUERIES[i]), CLIPV[i]], 1)),
     "live_compute_only": lambda i: head(torch.cat([enc_text(QUERIES[i]), enc_img_tensor(PRE[i])], 1)),
     "live_e2e_with_image_io": lambda i: head(torch.cat([enc_text(QUERIES[i]), enc_img_tensor(clip_pre(Image.open(PATHS[i]).convert("RGB")).unsqueeze(0).to(dev))], 1))}
RAWLAT, SUMLAT = [], []
for a, head in HEADS.items():
    for mode, fn in modes(head).items():
        ts = timeit(fn); RAWLAT += [dict(head=a, mode=mode, iter=i, ms=float(t)) for i, t in enumerate(ts)]
        SUMLAT.append(dict(head=a, params=PARAMS[a], mode=mode, n=len(ts), mean_ms=ts.mean(), sd_ms=ts.std(ddof=1), median_ms=np.median(ts), p5_ms=np.percentile(ts, 5), p95_ms=np.percentile(ts, 95)))
        print(f"{a:7s} {mode:24s} mean {ts.mean():7.3f} ms  sd {ts.std(ddof=1):6.3f}  p95 {np.percentile(ts,95):7.3f}", flush=True)
LAT = pd.DataFrame(SUMLAT); pd.DataFrame(RAWLAT).to_csv(D05 / "canonical_latency_raw.csv", index=False); LAT.to_csv(D05 / "canonical_latency_summary.csv", index=False)
ENV = dict(gpu=GPU_NAME, torch=torch.__version__, cuda=torch.version.cuda, batch_size=1, inference_mode=True, n_pairs=N_PAIRS, warmup=N_WARM, timed=N_TIMED,
           inputs="real human queries + real COCO images (known/new pools), random_state=0 sample of 200 pairs", checkpoints="Mixed-trained, seed 42 (KAN/MLP canonical; MLP-PM/Linear from NB1)")
json.dump(ENV, open(D05 / "canonical_latency_environment.json", "w"), indent=2)
# comparison with the LLM routers - only if the two measurements share a hardware/software signature
llm_env_p, llm_lat_p = D04 / "llm_router_latency_environment.json", D04 / "llm_router_latency.csv"
LLMLAT = pd.read_csv(llm_lat_p) if llm_lat_p.exists() else pd.DataFrame()
llm_env = json.load(open(llm_env_p)) if llm_env_p.exists() else {}
SAME_ENV = bool(llm_env) and llm_env.get("gpu") == GPU_NAME and llm_env.get("torch") == torch.__version__
rows = []
for a in HEADS:
    r = LAT[LAT["head"] == a].set_index("mode").mean_ms
    rec = dict(head=a, params=PARAMS[a], router_only_ms=r["router_only"], cached_image_ms=r["cached_image"], live_compute_only_ms=r["live_compute_only"], live_e2e_ms=r["live_e2e_with_image_io"],
               router_share_of_live_pct=100 * r["router_only"] / r["live_e2e_with_image_io"])
    for _, l in LLMLAT.iterrows(): rec[f"speedup_vs_{l['model']}"] = (l["mean_ms"] / r["live_e2e_with_image_io"]) if SAME_ENV else np.nan
    rows.append(rec)
LATTAB = pd.DataFrame(rows); LATTAB.to_csv(D05 / "latency_table_final.csv", index=False)
(D05 / "LATENCY_PROTOCOL.md").write_text("# Canonical latency protocol\n\n" + json.dumps(ENV, indent=2) +
    "\n\nModes: router_only = forward pass on a precomputed real 896-d vector; cached_image = live MiniLM + cached CLIP vector + head; live_compute_only = live MiniLM + CLIP image encoder on an already-preprocessed GPU tensor + head; "
    "live_e2e_with_image_io = additionally JPEG decode + CLIP preprocessing from disk. Each timing: 50 warm-up + 500 timed iterations, CUDA-synchronised wall clock, torch.inference_mode (no gradient tracking). "
    "Router-only latency is NOT end-to-end latency. Encoder times are shared by all heads; per-encoder times can only be derived by subtraction.\n\n"
    f"LLM comparison: NB2 measured {', '.join(LLMLAT['model']) if len(LLMLAT) else 'n/a'} at batch 1 in 4-bit NF4. Same hardware+torch signature as this run: {SAME_ENV}. "
    f"{'Speed-up ratios are therefore reported.' if SAME_ENV else 'Signatures differ, so NO direct speed-up ratio is computed; compare the absolute numbers only.'}\n")
print(LATTAB.round(3).to_string(index=False)); print("same environment signature as NB2:", SAME_ENV)


In [ ]:
# ===== NB3 | Section 8: verify KAN edge-function statistics from the nine canonical checkpoints (200-point grid on [-2,2]) =====
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
CKD = JR / "checkpoints_seed_corrected"; rows, R2 = [], {0: [], 1: []}
for regime in ("qwen", "llama", "mixed"):
    for seed in SEEDS:
        sdn = load_sd_np(CKD / f"kan_{regime}_shared28_seed{seed}_corrected_best.pt"); rec = dict(regime=regime, seed=seed)
        for li in (0, 1):
            st = kan_edge_stats(sdn[f"kan.layers.{li}.spline_linear.weight"], 48); R2[li].append(st["r2"])
            for k, v in edge_summary(st).items(): rec[f"L{li+1}_{k}"] = v
        rows.append(rec)
EDGE = pd.DataFrame(rows); EDGE.to_csv(D07 / "kan_edge_statistics_per_model.csv", index=False)
def ms(col, scale=100): return f"{EDGE[col].mean()*scale:.1f} ({EDGE[col].min()*scale:.1f}-{EDGE[col].max()*scale:.1f})"
summ = pd.DataFrame([
    {"Statistic": "Edges per model", "Layer 1": int(EDGE.L1_edges.iloc[0]), "Layer 2": int(EDGE.L2_edges.iloc[0])},
    {"Statistic": "Near-linear edges (R2>=0.95), %", "Layer 1": ms("L1_near_linear_share"), "Layer 2": ms("L2_near_linear_share")},
    {"Statistic": "Median R2 of linear fit", "Layer 1": ms("L1_median_r2", 1).replace(".6", ".63") if False else f"{EDGE.L1_median_r2.mean():.2f} ({EDGE.L1_median_r2.min():.2f}-{EDGE.L1_median_r2.max():.2f})", "Layer 2": f"{EDGE.L2_median_r2.mean():.2f} ({EDGE.L2_median_r2.min():.2f}-{EDGE.L2_median_r2.max():.2f})"},
    {"Statistic": "Monotone edges, %", "Layer 1": ms("L1_monotone_share"), "Layer 2": ms("L2_monotone_share")},
    {"Statistic": "Median peak-to-peak amplitude", "Layer 1": f"{EDGE.L1_median_amp.mean():.2f} ({EDGE.L1_median_amp.min():.2f}-{EDGE.L1_median_amp.max():.2f})", "Layer 2": f"{EDGE.L2_median_amp.mean():.2f} ({EDGE.L2_median_amp.min():.2f}-{EDGE.L2_median_amp.max():.2f})"},
    {"Statistic": "Share of a tool's amplitude in top-3 of 48 edges, %", "Layer 1": "-", "Layer 2": ms("L2_top3_amp_share")},
    {"Statistic": "Share of a tool's amplitude in top-10 of 48 edges, %", "Layer 1": "-", "Layer 2": ms("L2_top10_amp_share")}])
summ.to_csv(D07 / "kan_edge_summary.csv", index=False); (D07 / "table_kan_edge_stats.tex").write_text(latex_table(summ, "Learned KAN edge functions in the nine canonical KAN models: mean (min--max).", "tab:kan_edges"))
print(summ.to_string(index=False))

def edge_curve(W, in_dim, o, i, G=4, lo=-2.0, hi=2.0, npts=200):
    v = np.linspace(lo, hi, npts); c = np.linspace(lo, hi, G); rho = (hi - lo) / (G - 1)
    return v, np.exp(-(((v[:, None] - c[None, :]) / rho) ** 2)) @ W.reshape(W.shape[0], in_dim, G)[o, i]
sdm = load_sd_np(CKD / "kan_mixed_shared28_seed42_corrected_best.pt"); W2 = sdm["kan.layers.1.spline_linear.weight"]; st2 = kan_edge_stats(W2, 48)
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].hist(np.concatenate(R2[0]), bins=40, alpha=.6, density=True, label="layer 1"); axs[0].hist(np.concatenate(R2[1]), bins=40, alpha=.6, density=True, label="layer 2")
axs[0].axvline(0.95, ls=":", c="k"); axs[0].set_xlabel("R2 of best linear fit to s_io on [-2,2]"); axs[0].set_title("(a) edge non-linearity (9 models)"); axs[0].legend()
for ax, tool, lab in ((axs[1], "depth_estimation", "(b)"), (axs[2], "image_captioning", "(c)")):
    o = TOOL2ID[tool]
    for i in np.argsort(-st2["amp"][o])[:3]:
        v, s = edge_curve(W2, 48, o, int(i)); ax.plot(v, s, label=f"hidden unit {int(i)}")
    ax.axhline(0, c="gray", lw=.8); ax.set_title(f"{lab} {tool}: top-3 layer-2 edges"); ax.set_xlabel("layer-normalised input"); ax.legend()
fig.tight_layout(); fig.savefig(D07 / "fig_kan_edge_functions.png", dpi=150); plt.close(fig)
(D07 / "supplementary_legacy_analysis").mkdir(exist_ok=True)
(D07 / "supplementary_legacy_analysis" / "README.md").write_text("EXPLORATORY / EARLIER-STAGE / NON-CANONICAL.\nThe 35-class feature ablation, the random-row-split latency run and the 60-case gradient audit are not reproduced here and must not be used as main evidence.\n")


In [ ]:
# ===== NB3 | Section 6: independent-annotation kit (+ agreement analysis if annotator files are attached) =====
h = hb.h.copy(); order = np.random.RandomState(7).permutation(len(h))
blind = pd.DataFrame(dict(query_id=h.query_id.values[order], query_text=h["query"].values[order], image_reference=[f"{i}.jpg" for i in h.image_id.values[order]]))
blind.to_csv(D06 / "human_annotation_blinded.csv", index=False)
pd.DataFrame(dict(query_id=blind.query_id, chosen_tool="", is_ambiguous="", acceptable_tool_labels="", routing_requires_visual_context="", comments="")).to_csv(D06 / "human_annotation_response_template.csv", index=False)
try:                                                     # flat image folder (no known/new subfolders, so the pool is not revealed)
    KD, ND = [next(p for p in Path("/kaggle/input").rglob(n) if p.is_dir() and any(p.glob("*.jpg"))) for n in ("known", "new")]
    (D06 / "annotation_images").mkdir(exist_ok=True); copied = 0
    for i in sorted(set(h.image_id)):
        for d in (KD, ND):
            if (d / f"{i}.jpg").exists(): shutil.copy(d / f"{i}.jpg", D06 / "annotation_images" / f"{i}.jpg"); copied += 1; break
    print("annotation images copied:", copied, "of", h.image_id.nunique())
except StopIteration:
    print("images dataset not attached - annotation_images/ not created (attach images-new-known to create it)")

guide = ["# Human Annotation Guide (independent annotators)\n",
 "You are asked to label 372 short user requests, each shown with a COCO image. You must NOT have written or seen the original labels, and you must not discuss items with anyone or use any AI system while labelling. Work alone.\n",
 "## Task - for every row of `human_annotation_blinded.csv` (image: `annotation_images/<image_reference>`) fill one row of `human_annotation_response_template.csv`\n",
 "1. `chosen_tool`: the single best tool for the request, using exactly one name from the list below.",
 "2. `is_ambiguous`: `yes` if more than one tool from the list could legitimately serve the request, otherwise `no`.",
 "3. `acceptable_tool_labels`: all tools that would be acceptable (comma-separated, always including `chosen_tool`); if not ambiguous, just repeat `chosen_tool`.",
 "4. `routing_requires_visual_context`: `yes` if you could NOT choose the tool from the wording alone and had to look at the image; `no` otherwise.",
 "5. `comments`: optional (unclear wording, wrong/missing tool, etc.).\n",
 "Judge only which TOOL should handle the request, not whether the request is well written. Do not try to guess an original label.\n",
 "## The 28 tools\n"] + [f"- `{k}`: {v}" for k, v in TOOL_DEFS.items()] + ["\n## Boundary notes (when two tools are easily confused)\n"] + [f"- {b}" for b in BOUNDARY_NOTES] + [
 "\n## Returning your work\n", "Save your file as `human_annotation_annotator<N>.csv` (N = your number), UTF-8 CSV, same columns and rows as the template. Expected effort: about 2-3 hours.",
 "\nThe tool names and one-line definitions above are the same text that was shown to the zero-shot LLM baselines; they were written before any annotation or LLM result existed."]
(D06 / "HUMAN_ANNOTATION_GUIDE.md").write_text("\n".join(guide))
adj = pd.DataFrame(dict(query_id=h.query_id, original_author_label=h.tool_name, original_acceptable_set=h.acceptable_tool_labels, original_is_ambiguous=np.where(hb.is_amb, "yes", "no"),
                        original_visual_context=h.routing_requires_visual_context, annotator_1_label="", annotator_2_label="", annotator_1_set="", annotator_2_set="",
                        annotator_1_ambiguous="", annotator_2_ambiguous="", annotator_1_visual="", annotator_2_visual="",
                        final_adjudicated_label="", final_acceptable_set="", final_visual_context="", adjudication_note=""))

ann_files = sorted(Path("/kaggle/input").rglob("human_annotation_annotator*.csv")); VALIDATION = dict(status="WAITING_FOR_INDEPENDENT_ANNOTATION", n_annotators=0)
if ann_files:
    A = {}
    for f in ann_files:
        d = pd.read_csv(f).fillna(""); need = {"query_id", "chosen_tool", "is_ambiguous", "acceptable_tool_labels", "routing_requires_visual_context"}
        probs = []
        if not need.issubset(d.columns): probs.append(f"missing columns {sorted(need - set(d.columns))}")
        else:
            if set(d.query_id) != set(h.query_id): probs.append("query_id set differs from the benchmark")
            bad = sorted(set(d.chosen_tool) - TOOLS)
            if bad: probs.append(f"unknown tool names: {bad[:5]}")
        print(f.name, "->", "OK" if not probs else probs)
        if not probs: A[f.stem] = d
    VALIDATION = dict(status="ANNOTATOR_FILES_FOUND_BUT_INVALID", n_annotators=0, files=[f.name for f in ann_files])
    names = sorted(A)
    if names:
        auth = pd.DataFrame(dict(query_id=h.query_id, chosen_tool=h.tool_name, is_ambiguous=np.where(hb.is_amb, "yes", "no"), acceptable_tool_labels=h.acceptable_tool_labels, routing_requires_visual_context=h.routing_requires_visual_context))
        reps, disag = [], []
        for n in names:
            rep, m = agreement_report(auth, A[n]); rep["pair"] = f"authors_vs_{n}"; reps.append(rep); disag.append(m[m.chosen_tool_1 != m.chosen_tool_2].assign(pair=rep["pair"]))
        for a_i in range(len(names)):
            for b_i in range(a_i + 1, len(names)):
                rep, m = agreement_report(A[names[a_i]], A[names[b_i]]); rep["pair"] = f"{names[a_i]}_vs_{names[b_i]}"; reps.append(rep); disag.append(m[m.chosen_tool_1 != m.chosen_tool_2].assign(pair=rep["pair"]))
        AG = pd.DataFrame(reps); AG.to_csv(D06 / "human_annotation_agreement.csv", index=False); pd.concat(disag).to_csv(D06 / "human_annotation_disagreements.csv", index=False)
        for k, n in enumerate(names[:2], 1):
            d = A[n].set_index("query_id").loc[adj.query_id]
            adj[f"annotator_{k}_label"], adj[f"annotator_{k}_set"] = d.chosen_tool.values, d.acceptable_tool_labels.values
            adj[f"annotator_{k}_ambiguous"], adj[f"annotator_{k}_visual"] = d.is_ambiguous.values, d.routing_requires_visual_context.values
        (D06 / "HUMAN_ANNOTATION_AGREEMENT.md").write_text("# Human annotation agreement (computed)\n\n" + AG.round(3).to_markdown(index=False) +
            "\n\nThe adjudication file leaves every `final_*` column blank: the final label is a human decision. Until an adjudicated benchmark exists, the benchmark must not be described as independently validated.\n")
        VALIDATION = dict(status="ANNOTATION_AGREEMENT_COMPUTED_ADJUDICATION_PENDING", n_annotators=len(names), files=[f.name for f in ann_files])
adj.to_csv(D06 / "human_annotation_adjudication_template.csv", index=False)
if VALIDATION["n_annotators"] == 0:
    (D06 / "HUMAN_ANNOTATION_AGREEMENT.md").write_text("# Human annotation agreement\n\nSTATUS: waiting for independent annotators. No agreement statistic exists yet; none may be claimed.\n"
        "Send `human_annotation_blinded.csv`, `annotation_images/` and `HUMAN_ANNOTATION_GUIDE.md` to >= 2 people who did not write the benchmark; collect `human_annotation_annotator<N>.csv`; re-run this cell with those files attached.\n")
adjud = list(Path("/kaggle/input").rglob("human_shared28_independently_validated.csv"))
if adjud and VALIDATION["n_annotators"] > 0:
    hb2 = HumanBench(adjud[0], JR / "qwen_shared28_train.csv", JR / "llama_shared28_train.csv", JR / "shared_28_class_mapping.json"); rr = []
    for a, regime, seed in [(a, r, s) for a in ("kan", "mlp") for r in ("qwen", "llama", "mixed") for s in SEEDS]:
        df = read_pred(JR / "part4_predictions_seed_corrected" / f"{a}_{regime}_seed{seed}_on_human.csv")
        for r in hb2.eval_human(df, use_bench_labels=True): r.update(arch=a, regime=regime, seed=seed); rr.append(r)
    pd.DataFrame(rr).to_csv(D06 / "human_results_recomputed_after_adjudication.csv", index=False); VALIDATION["status"] = "ADJUDICATED_BENCHMARK_FOUND_RESULTS_RECOMPUTED"
json.dump(VALIDATION, open(D06 / "validation_status.json", "w"), indent=2); print(VALIDATION)


In [ ]:
# ===== NB3 | Section 10: paired statistics on the saved predictions (per-seed McNemar, paired bootstrap, Holm) =====
def pred_file(model, regime, seed, ev):
    if model in ("kan", "mlp"): return JR / "part4_predictions_seed_corrected" / f"{model}_{regime}_seed{seed}_on_{ev}.csv"
    feat, arch = {"kan_text": ("text_only", "kan"), "mlp_text": ("text_only", "mlp"), "mlp_pm": ("text_pixel", "mlp_pm"), "linear": ("text_pixel", "linear")}[model]
    d = {"kan": D01, "mlp": D01, "mlp_pm": D02, "linear": D03}[arch]
    return d / "predictions" / f"{feat}_{arch}_{regime}_seed{seed}_on_{ev}.csv"
_cache = {}
def get_yp(model, regime, seed, ev):
    k = (model, regime, seed, ev)
    if k not in _cache:
        d = pd.read_csv(pred_file(*k), usecols=["ground_truth", "pred_top1"]); _cache[k] = (d.ground_truth.values.astype(int), d.pred_top1.values.astype(int))
    return _cache[k]
CELLS = {"qwen_test": ("qwen_test", None), "llama_test": ("llama_test", None), "qwen_test_disjoint": ("qwen_test", sb.disj["qwen_test"]),
         "llama_test_disjoint": ("llama_test", sb.disj["llama_test"]), "human_all": ("human", None), "human_new": ("human", ~hb.known), "human_vis_yes": ("human", hb.vis_yes)}
COMPS = [("kan", "mlp"), ("kan", "mlp_pm"), ("kan", "linear"), ("mlp", "mlp_pm"), ("mlp_pm", "linear"), ("kan", "kan_text"), ("mlp", "mlp_text")]
rows = []
for a, b in COMPS:
    fam = []
    for regime in ("qwen", "llama", "mixed"):
        for cell, (ev, mask) in CELLS.items():
            ys, pa, pb, ps, bc = None, [], [], [], []
            for s in SEEDS:
                y, p1 = get_yp(a, regime, s, ev); _, p2 = get_yp(b, regime, s, ev)
                if mask is not None: y, p1, p2 = y[mask], p1[mask], p2[mask]
                ys = y; pa.append(p1); pb.append(p2); bb, cc, pv = mcnemar_exact(p1 == y, p2 == y); ps.append(pv); bc.append((bb, cc))
            (l1, h1), (l2, h2) = paired_bootstrap(pa, pb, ys, B=1000, seed=0)
            d1 = float(np.mean([(p1 == ys).mean() - (p2 == ys).mean() for p1, p2 in zip(pa, pb)])); d2 = float(np.mean([macro_f1_fast(ys, p1, ALL_LABELS) - macro_f1_fast(ys, p2, ALL_LABELS) for p1, p2 in zip(pa, pb)]))
            fam.append(dict(model_a=a, model_b=b, regime=regime, cell=cell, n=len(ys), d_top1_pp=d1 * 100, d_top1_ci_lo=l1 * 100, d_top1_ci_hi=h1 * 100, d_macro_f1_pp=d2 * 100, d_macro_f1_ci_lo=l2 * 100, d_macro_f1_ci_hi=h2 * 100,
                            mcnemar_p_seed42=ps[0], mcnemar_p_seed123=ps[1], mcnemar_p_seed2026=ps[2], mcnemar_p_max_over_seeds=max(ps), seeds_a_better=int(sum(1 for (bb, cc) in bc if bb > cc))))
    adj_p = holm([r["mcnemar_p_max_over_seeds"] for r in fam])
    for r, q in zip(fam, adj_p): r["holm_p"] = q; r["significant_holm_0.05"] = bool(q < 0.05 and r["seeds_a_better"] in (0, 3))
    rows += fam
STAT = pd.DataFrame(rows); STAT.to_csv(D08 / "statistical_tests.csv", index=False)
summ = STAT.groupby(["model_a", "model_b"]).agg(tests=("cell", "size"), significant=("significant_holm_0.05", "sum")).reset_index()
md = ["# Statistical analysis (computed)\n",
      "Per seed, an exact McNemar test compares the two heads' Top-1 correctness on identical test rows. The reported p is the MAXIMUM over the three seeds (a result counts only if it holds for every seed), Holm-corrected within each comparison "
      "(family = 3 regimes x 7 cells). A test is flagged significant only if Holm p < 0.05 AND the same model is better in all 3 seeds. Paired-bootstrap intervals (1,000 resamples of test rows, shared across seeds, seed-averaged difference) "
      "describe TEST-SAMPLING variance only; seed variance is shown by the per-seed columns. With three seeds none of this is a substitute for more training runs.\n",
      "Rows of one test set are not independent (Qwen paraphrase families), so synthetic-set p-values are optimistic; the 372-query human set is the cleaner place to read significance.\n", "## Holm-significant tests per comparison (A minus B)\n", summ.to_markdown(index=False),
      "\n## Human-written (all 372), seed-averaged differences\n"]
hh = STAT[STAT.cell == "human_all"][["model_a", "model_b", "regime", "d_top1_pp", "d_top1_ci_lo", "d_top1_ci_hi", "d_macro_f1_pp", "mcnemar_p_max_over_seeds", "holm_p", "seeds_a_better"]]
md.append(hh.round(3).to_markdown(index=False)); (D08 / "STATISTICAL_ANALYSIS_FINAL.md").write_text("\n".join(md)); print(summ.to_string(index=False))

# ---- zero-shot LLM vs trained heads on the human set (Mixed-trained heads, per-seed McNemar, p = max over seeds) ----
LLMV = []
for m in ("qwen2.5-7b-instruct", "llama-3.1-8b-instruct"):
    pth = D04 / f"parsed_predictions_{m}_query_only_human.csv"
    if not pth.exists(): continue
    det = pd.read_csv(pth).fillna(""); assert (det.sample_id.astype(str).values == hb.h.query_id.astype(str).values).all()
    cl = (det.parsed.values == det.gold_tool.values)
    for head in ("kan", "mlp", "mlp_pm", "linear"):
        ps, d1 = [], []
        for s in SEEDS:
            y, p = get_yp(head, "mixed", s, "human"); ch = (p == y); _, _, pv = mcnemar_exact(cl, ch); ps.append(pv); d1.append(cl.mean() - ch.mean())
        LLMV.append(dict(llm=m, head=head, llm_top1_pct=cl.mean() * 100, llm_minus_head_pp=float(np.mean(d1)) * 100, mcnemar_p_max_over_seeds=max(ps)))
LLMV = pd.DataFrame(LLMV); LLMV.to_csv(D08 / "llm_vs_heads_human_mcnemar.csv", index=False); print(LLMV.round(4).to_string(index=False))


In [ ]:
# ===== NB3 | Section 9: one final comparison table (trained heads + zero-shot LLMs) =====
RES1 = pd.read_csv(ALLCELLS); RP = RES1[RES1.feat == "text_pixel"].copy()
LLM = pd.read_csv(D04 / "llm_router_results.csv"); LLM3 = pd.read_csv(D04 / "llm_router_human_top3.csv") if (D04 / "llm_router_human_top3.csv").exists() else pd.DataFrame()
LATF = pd.read_csv(D05 / "latency_table_final.csv").set_index("head") if (D05 / "latency_table_final.csv").exists() else pd.DataFrame()
def vals(arch, pairs, col):
    return np.array([np.mean([float(RP[(RP.arch == arch) & (RP.regime == r) & (RP.cell == c) & (RP.seed == s)][col].iloc[0]) for r, c in pairs]) for s in SEEDS])
def s_(v): return pm(v.mean(), sd(v))
NAMES = {"linear": "Linear head", "mlp_pm": "Parameter-matched MLP", "mlp": "MLP (original)", "kan": "KAN-Route"}
PAR = {"linear": 25116, "mlp_pm": 62003, "mlp": 95708, "kan": 61660}
rows = []
for a in ("linear", "mlp_pm", "mlp", "kan"):
    mx = lambda c, col: vals(a, [("mixed", c)], col)
    rows.append({"Model": NAMES[a], "Input": "MiniLM + CLIP-pixel (896-d)", "Trainable params": f"{PAR[a]:,}",
        "Qwen in-domain Top-1": s_(vals(a, [("qwen", "qwen_test")], "top1")), "Llama in-domain Top-1": s_(vals(a, [("llama", "llama_test")], "top1")),
        "Cross-teacher Top-1": s_(vals(a, [("qwen", "llama_test"), ("llama", "qwen_test")], "top1")), "Image-disjoint cross Top-1": s_(vals(a, [("qwen", "llama_test_disjoint"), ("llama", "qwen_test_disjoint")], "top1")),
        "Human all Top-1*": s_(mx("human_all", "top1")), "Human strict Top-1*": s_(mx("human_all", "strict_top1")), "Human strict Top-3*": s_(mx("human_all", "strict_top3")),
        "Human ambig-aware Top-1*": s_(mx("human_all", "ambig_top1")), "Human macro-F1*": s_(mx("human_all", "macro_f1")), "Human new-image Top-1*": s_(mx("human_new", "top1")),
        "Router-only ms": f"{LATF.loc[a,'router_only_ms']:.2f}" if a in LATF.index else "n/a", "Live e2e ms": f"{LATF.loc[a,'live_e2e_ms']:.2f}" if a in LATF.index else "n/a"})
LAT_LLM = pd.read_csv(D04 / "llm_router_latency.csv").set_index("model") if (D04 / "llm_router_latency.csv").exists() else pd.DataFrame()
def lf(m, cell, col, cond="query_only"):
    r = LLM[(LLM.model == m) & (LLM.condition == cond) & (LLM.cell == cell)]; return float(r[col].iloc[0]) if len(r) else np.nan
for m, nm, pb in (("qwen2.5-7b-instruct", "Zero-shot Qwen2.5-7B-Instruct", "7.6B"), ("llama-3.1-8b-instruct", "Zero-shot Llama-3.1-8B-Instruct", "8.0B")):
    if m not in set(LLM.model): continue
    t3 = LLM3[(LLM3.model == m) & (LLM3.cell == "human_all")].strict_top3 if len(LLM3) else pd.Series(dtype=float)
    rows.append({"Model": nm, "Input": "Query text only (+object list: extra run)", "Trainable params": f"0 (frozen {pb})",
        "Qwen in-domain Top-1": f"{lf(m,'qwen_test','top1')*100:.2f}$^\\dagger$", "Llama in-domain Top-1": f"{lf(m,'llama_test','top1')*100:.2f}$^\\dagger$", "Cross-teacher Top-1": "N/A",
        "Image-disjoint cross Top-1": f"{np.nanmean([lf(m,'qwen_test_disjoint','top1'), lf(m,'llama_test_disjoint','top1')])*100:.2f}$^\\dagger$",
        "Human all Top-1*": f"{lf(m,'human_all','top1')*100:.2f}", "Human strict Top-1*": f"{lf(m,'human_all','strict_top1')*100:.2f}", "Human strict Top-3*": f"{float(t3.iloc[0])*100:.2f}" if len(t3) else "n/a",
        "Human ambig-aware Top-1*": f"{lf(m,'human_all','ambig_top1')*100:.2f}", "Human macro-F1*": f"{lf(m,'human_all','macro_f1')*100:.2f}", "Human new-image Top-1*": f"{lf(m,'human_new','top1')*100:.2f}",
        "Router-only ms": "N/A", "Live e2e ms": f"{LAT_LLM.loc[m,'mean_ms']:.0f} (LLM generation)" if m in LAT_LLM.index else "n/a"})
FINAL = pd.DataFrame(rows); FINAL.to_csv(D09 / "final_comparison_table.csv", index=False)
(D09 / "final_comparison_table.md").write_text(FINAL.to_markdown(index=False) + "\n\n*Human columns: Mixed-trained heads (best supervision), mean +/- SD over 3 seeds; LLM rows are single deterministic runs. $^\\dagger$ zero-shot score on that corpus, no training; the Qwen/Llama LLM scoring its own teacher corpus is not an independent measurement. N/A = not meaningful.\n")
(D09 / "table_final_comparison.tex").write_text(latex_table_resized(FINAL.drop(columns=["Input"]), "Final comparison. Human columns use Mixed-trained heads. Mean $\\pm$ SD over three seeds for trained heads; single deterministic run for zero-shot LLMs. $^\\dagger$: zero-shot, no training.", "tab:final_comparison"))
print(FINAL.drop(columns=["Input"]).to_string(index=False))


In [ ]:
# ===== NB3 | Sections 11-12: report, claim changes and the readiness gate (every check is computed) =====
checks = []
def chk(name, ok, detail): checks.append((name, bool(ok), detail))
chk("canonical Shared-28 evidence unchanged (hash re-check)", RECHECK["canonical_unchanged"], f"{RECHECK['files_checked']} files, mismatches={len(RECHECK['hash_mismatches'])}")
chk("leakage audit zero (recomputed)", RECHECK["leakage_zero"], f"qwen={RECHECK['qwen_overlaps']} llama={RECHECK['llama_overlaps']}")
abl = RES1[RES1.arch.isin(["kan", "mlp"]) & RES1.feat.isin(["text_only", "text_pixel"])]
chk("text-only vs text+pixel: 36 configurations x 3 eval sets", abl.groupby(["feat", "arch", "regime", "seed"]).ngroups == 36 and (abl.cell == "human_vis_yes").any(), f"{abl.groupby(['feat','arch','regime','seed']).ngroups} configurations; visual-context stratum present={(abl.cell=='human_vis_yes').any()}")
spec = json.load(open(D02 / "mlp_pm_architecture.json")); pm_rel = abs(spec["mlp_param_matched"]["relative_to_kan_pct"])
chk("parameter-matched MLP: 9 models, parameters within 2% of KAN", RES1[RES1.arch == "mlp_pm"].groupby(["regime", "seed"]).ngroups == 9 and pm_rel <= 2, f"{spec['mlp_param_matched']['params']:,} vs {spec['kan']['params']:,} ({pm_rel:.2f}%)")
chk("linear baseline: 9 models", RES1[RES1.arch == "linear"].groupby(["regime", "seed"]).ngroups == 9, "9 expected")
chk("zero-shot LLM: both models complete on human + both teacher test sets", all(m in set(LLM.model) and {"qwen_test", "llama_test", "human_all"} <= set(LLM[LLM.model == m].cell) for m in ("qwen2.5-7b-instruct", "llama-3.1-8b-instruct")), f"models={sorted(set(LLM.model))}")
chk("canonical latency re-measured (4 heads x 4 modes)", (D05 / "canonical_latency_summary.csv").exists() and len(pd.read_csv(D05 / "canonical_latency_summary.csv")) == 16, "canonical_latency_summary.csv")
chk("KAN edge statistics verified on 9 canonical models", len(EDGE) == 9, f"{len(EDGE)} models")
worst = 0.0
for (f_, a, r, s), g in RES1.groupby(["feat", "arch", "regime", "seed"]):
    d = {"kan": D01, "mlp": D01, "mlp_pm": D02, "linear": D03}[a]
    for ev, cell in (("qwen_test", "qwen_test"), ("llama_test", "llama_test"), ("human", "human_all")):
        p = pd.read_csv(d / "predictions" / f"{f_}_{a}_{r}_seed{s}_on_{ev}.csv", usecols=["correct"]).correct.mean()
        worst = max(worst, abs(p - float(g[g.cell == cell].top1.iloc[0])))
chk("new metrics reproduce from raw prediction files", worst < 1e-9, f"max |Top-1 recompute - table| = {worst:.2e} over 54 configurations x 3 sets")
by_design = ["strict_n", "strict_top1", "strict_top3", "ambig_n", "ambig_top1", "ambig_top3", "macro_f1_shared26", "top3", "macro_f1_label_set"]
core, stb = RES1[~RES1.cell.str.endswith("stable21")], RES1[RES1.cell.str.endswith("stable21")]
core_cols = [c for c in core.select_dtypes("number").columns if c not in by_design]
fin_ok = np.isfinite(core[core_cols].to_numpy(float)).all() and np.isfinite(stb[["n", "top1", "top3", "macro_f1"]].to_numpy(float)).all()
chk("no NaN/Inf in primary metrics", fin_ok, "strict/ambiguity/Top-3 columns are NaN by design where they do not apply; weighted-F1 is not defined for the stable-support cells")
chk("statistical tests computed", (D08 / "statistical_tests.csv").exists(), f"{len(STAT)} tests")
chk("annotation kit generated", (D06 / "human_annotation_blinded.csv").exists() and (D06 / "HUMAN_ANNOTATION_GUIDE.md").exists(), "blinded csv + guide + response/adjudication templates")
auto_ok = all(ok for _, ok, _ in checks)
HUMAN_STATUS = VALIDATION["status"]; human_done = HUMAN_STATUS in ("ADJUDICATED_BENCHMARK_FOUND_RESULTS_RECOMPUTED",)
EVID = "YES" if (auto_ok and human_done) else "NO"
gate = ["# Neurocomputing code-readiness gate (computed)\n", "| Check | Result | Evidence |", "|---|---|---|"] + [f"| {n} | {'PASS' if ok else 'FAIL'} | {d} |" for n, ok, d in checks]
gate += [f"\nAUTOMATED_EXPERIMENTS_READY = {'YES' if auto_ok else 'NO'}", f"HUMAN_VALIDATION_STATUS = {HUMAN_STATUS}",
         f"NEUROCOMPUTING_SUBMISSION_EVIDENCE_READY = {EVID}" + ("" if EVID == "YES" else "  (blocked only by the missing independent annotation / adjudication; no code can remove this)" if auto_ok else "  (an automated check failed)")]
(D09 / "NEUROCOMPUTING_CODE_READINESS.md").write_text("\n".join(gate)); print("\n".join(gate))

# ---------------- claim changes, derived from the numbers ----------------
cmp_pm = pd.read_csv(D02 / "mlp_pm_comparison_to_kan.csv")
def gp(a, b, grp, m, scope="all28"):
    r = cmp_pm[(cmp_pm.a == a) & (cmp_pm.b == b) & (cmp_pm.group == grp) & (cmp_pm.metric == m) & (cmp_pm.scope == scope)].iloc[0]; return r
hw = gp("kan", "mlp_pm", "human_written", "top1"); hf1 = gp("kan", "mlp_pm", "human_written", "macro_f1"); hs = gp("kan", "mlp_pm", "human_written", "macro_f1", "stable21")
best_row = FINAL[FINAL.Model.isin(NAMES.values())].iloc[:, :]; kan_h = vals("kan", [("mixed", "human_all")], "top1"); mlp_h = vals("mlp", [("mixed", "human_all")], "top1")
best_trained = max(kan_h.mean(), mlp_h.mean()); llm_h = {m: lf(m, "human_all", "top1") for m in ("qwen2.5-7b-instruct", "llama-3.1-8b-instruct") if m in set(LLM.model)}
def rd(p):
    return Path(p).read_text() if Path(p).exists() else f"(not available: {Path(p).name})"
LLMV = pd.read_csv(D08 / "llm_vs_heads_human_mcnemar.csv") if (D08 / "llm_vs_heads_human_mcnemar.csv").exists() else pd.DataFrame()
pix = rd(D01 / "FEATURE_ABLATION_VERDICT.md")
changes = ["# Claim changes required in the manuscript (derived from the computed results)\n",
 f"1. **Pixel stream.** See FEATURE_ABLATION_VERDICT.md. Pooled Text+Pixel minus Text-only: " + "; ".join(l.strip("- ").split(":")[0] + " " + l.split("=")[1].split("(")[0].strip() for l in pix.splitlines() if l.startswith("- ") and ("Qwen test" in l or "Human-written" in l or "NEED" in l or "answerable" in l)) + ". Do not claim the visual stream improves routing unless those lines say 'improves'.",
 f"2. **KAN vs a size-matched MLP.** Parameters: KAN {PAR['kan']:,} vs MLP-PM {PAR['mlp_pm']:,} -> the old '35.6% fewer parameters' claim must be removed or restated as 'comparable size'. On human-written queries KAN minus MLP-PM Top-1 = {hw.diff_mean*100:+.2f} +/- {hw.diff_sd*100:.2f} pp (KAN ahead in {hw.seeds_a_ahead}/3 seeds), macro-F1 {hf1.diff_mean*100:+.2f} pp; on the 21 stable-support classes human macro-F1 {hs.diff_mean*100:+.2f} pp. The earlier 'comparable in Top-1' sentence is no longer safe for human-written queries; it must be rewritten from MLP-PM_VERDICT numbers, and the head-training-budget caveat (different lr/epochs) must stay.",
 f"2b. **Is the KAN-vs-MLP-PM human-set advantage statistically supported?** Per-regime exact McNemar on the 372 human queries, Holm-corrected, flagged only if all 3 seeds agree: significant in {int(STAT[(STAT.model_a=='kan')&(STAT.model_b=='mlp_pm')&(STAT.cell=='human_all')]['significant_holm_0.05'].sum())}/3 regimes (Holm p = " + ", ".join(f"{p:.3f}" for p in STAT[(STAT.model_a=='kan')&(STAT.model_b=='mlp_pm')&(STAT.cell=='human_all')].holm_p) + "). A consistent direction across seeds is not the same as a significant difference; word the claim accordingly.",
 f"2c. **Linear head.** A {PAR['linear']:,}-parameter linear head scores {vals('linear', [('mixed','human_all')], 'top1').mean()*100:.2f}% on human-written Top-1 and {vals('linear', [('mixed','human_all')], 'strict_top3').mean()*100:.2f}% strict Top-3, against KAN {vals('kan', [('mixed','human_all')], 'top1').mean()*100:.2f}% / {vals('kan', [('mixed','human_all')], 'strict_top3').mean()*100:.2f}%. KAN minus linear on human_all Top-1 is Holm-significant in {int(STAT[(STAT.model_a=='kan')&(STAT.model_b=='linear')&(STAT.cell=='human_all')]['significant_holm_0.05'].sum())}/3 regimes. If the linear head matches the non-linear heads on human wording, say so: head non-linearity is not what limits routing on human requests.",
 "3. **Zero-shot LLM routers.** " + "; ".join(f"{m}: {v*100:.2f}% human Top-1" for m, v in llm_h.items()) + f" vs best trained head (Mixed) {best_trained*100:.2f}%. " + (" Per-seed McNemar vs the Mixed heads (worst-case p over seeds): " + "; ".join(f"{r.llm[:5]} vs {r['head']}: {r.llm_minus_head_pp:+.1f} pp, p={r.mcnemar_p_max_over_seeds:.1e}" for _, r in LLMV.iterrows()) + ". " if len(LLMV) else "") +
 ("Zero-shot routers beat every trained head on human-written queries by a wide margin; the abstract, discussion and 'supervision matters more than the head' framing must say so explicitly, and the contribution should be stated as: cheap trained heads are ~100x faster but substantially less accurate on human wording." if llm_h and max(llm_h.values()) > best_trained + 0.05 else "Report the comparison as measured."),
 f"4. **Latency.** Use only 05_canonical_latency (real queries/images, T4, inference mode); remove the earlier random-tensor numbers. LLM latency comparison is a ratio only if the signature matches (see LATENCY_PROTOCOL.md; same signature = {SAME_ENV}).",
 "5. **Interpretability.** Edge statistics reproduce the manuscript table; keep 'inspectable edge functions' wording, never 'interpretable improves auditing' (no annotator study of explanations exists).",
 f"6. **Human benchmark.** Status: {HUMAN_STATUS}. Until independent annotators + adjudication exist, describe the benchmark as author-written and author-adjudicated only."]
(D09 / "CLAIM_CHANGES_REQUIRED.md").write_text("\n\n".join(changes)); print("\n\n".join(changes))
rep = ["# Neurocomputing Experiments Report (computed by NB1-NB3)\n", "## 1. Frozen baseline\n" + json.dumps(RECHECK, indent=1), "## 2. Canonical feature ablation\n" + pix,
       "## 3. Parameter-matched MLP\n" + rd(D02 / "PARAMETER_MATCHED_VERDICT.md"), "## 4. Linear baseline\n" + rd(D03 / "LINEAR_BASELINE_VERDICT.md"),
       "## 5. Zero-shot LLM routers\n" + rd(D04 / "ZERO_SHOT_LLM_BASELINE_SUMMARY.md"), "## 6. Final comparison\n" + rd(D09 / "final_comparison_table.md"),
       "## 7. Canonical latency\n" + rd(D05 / "LATENCY_PROTOCOL.md") + "\n" + LATTAB.round(3).to_markdown(index=False) if 'LATTAB' in globals() else "## 7. Canonical latency\nsee 05_canonical_latency",
       "## 8. KAN edge statistics\n" + summ_edges.to_markdown(index=False) if 'summ_edges' in globals() else "## 8. KAN edge statistics\n" + pd.read_csv(D07 / "kan_edge_summary.csv").to_markdown(index=False),
       "## 9. Statistics\n" + rd(D08 / "STATISTICAL_ANALYSIS_FINAL.md"), "## 10. Human validation\n" + rd(D06 / "HUMAN_ANNOTATION_AGREEMENT.md"), "## 11. Claim changes\n" + "\n\n".join(changes)]
(D09 / "NEUROCOMPUTING_EXPERIMENTS_REPORT.md").write_text("\n\n".join(rep))


In [ ]:
# ===== NB3 | Merge NB1 + NB2 + NB3 into one final package =====
for name, src in (("00_frozen_baseline", D00_NB1), ("01_canonical_feature_ablation", D01), ("02_parameter_matched_mlp", D02), ("03_linear_baseline", D03)):
    shutil.copytree(src, OUT / name, dirs_exist_ok=True)
shutil.copytree(D04, OUT / "04_zero_shot_llm", dirs_exist_ok=True)
for f in ("all_cells_by_seed_NB1.csv", "training_log_NB1.csv", "per_class_all_models_NB1.csv"):
    if (NB1_ROOT / f).exists(): shutil.copy(NB1_ROOT / f, OUT / f)
shutil.make_archive("/kaggle/working/KAN_ROUTE_NEUROCOMPUTING_FINAL_EXPERIMENTS", "zip", OUT)
print("DONE. Download /kaggle/working/KAN_ROUTE_NEUROCOMPUTING_FINAL_EXPERIMENTS.zip")
